In [3]:
import sys
from twisted.internet import asyncioreactor
# This installs the specific reactor before Scrapy tries to start its own
if "twisted.internet.reactor" in sys.modules:
    del sys.modules["twisted.internet.reactor"]
asyncioreactor.install()

In [4]:
import scrapy
from scrapy.crawler import CrawlerRunner # To Run our spider
import crochet
from scrapy.utils.reactor import install_reactor
crochet.setup()

output_data = list()

class SimpleSpider(scrapy.Spider):
    name='Simple'
    allowed_domains = ["www.example.com"]
    install_reactor("twisted.internet.asyncioreactor.AsyncioSelectorReactor")
    start_urls = [
            'http://quotes.toscrape.com/page/1',
            'http://quotes.toscrape.com/page/2',
            'http://quotes.toscrape.com/page/3'
        ]
    def start_requests(self):
        urls=[
            'http://quotes.toscrape.com/page/1',
            'http://quotes.toscrape.com/page/2',
            'http://quotes.toscrape.com/page/3'
        ]
        return [scrapy.Request(url=url,callback=self.parse) for url in urls]
    def parse(self,response):
        authors = response.xpath('//span/small/text()').extract()
        for author in authors:
            # 2. Append to the global list
            output_data.append(author)
            yield {'author': author}

process = CrawlerRunner({
    'USER_AGENT': 'Mozilla/4.0 (compatible; MSIE 7.0; Windows NT 5.1)',
})
process.crawl(SimpleSpider)

<Deferred at 0x232b1942490 current result: <twisted.python.failure.Failure builtins.RuntimeError: The installed reactor (twisted.internet.selectreactor.SelectReactor) does not match the requested one (twisted.internet.asyncioreactor.AsyncioSelectorReactor)>>

In [5]:
output_data

[]

In [2]:
import scrapy
from scrapy.crawler import CrawlerRunner
import crochet
import time

# 1. Initialize crochet
crochet.setup()

output_data = list()

class SimpleSpider(scrapy.Spider):
    name = 'SimpleSpider'

    def start_requests(self):
        urls = [
            'http://quotes.toscrape.com/page/1',
            'http://quotes.toscrape.com/page/2'
        ]
        for url in urls:
            yield scrapy.Request(url=url, callback=self.parse)

    def parse(self, response):
        authors = response.xpath('//span/small/text()').getall()
        for author in authors:
            output_data.append(author)
            yield {'author': author}

# 2. Configure and run
process = CrawlerRunner({
    'USER_AGENT': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
    'LOG_LEVEL': 'ERROR' # This hides the messy logs so you only see your output
})

# 3. Start the crawl
process.crawl(SimpleSpider)

# 4. Wait a few seconds for the spider to finish
print("Scraping in progress...")
time.sleep(5)

# 5. Display the output
print(f"\nSuccessfully captured {len(output_data)} authors:")
print(list(set(output_data))) # Use set() to show unique names

Exception in thread CrochetReactor:
Traceback (most recent call last):
  File "C:\Users\dell\anaconda3\Lib\threading.py", line 1043, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "C:\Users\dell\anaconda3\Lib\site-packages\ipykernel\ipkernel.py", line 772, in run_closure
    _threading_Thread_run(self)
    ~~~~~~~~~~~~~~~~~~~~~^^^^^^
  File "C:\Users\dell\anaconda3\Lib\threading.py", line 994, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\dell\anaconda3\Lib\site-packages\crochet\_eventloop.py", line 360, in <lambda>
    target=lambda: self._reactor.run(installSignalHandlers=False),
                   ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\dell\anaconda3\Lib\site-packages\twisted\internet\asyncioreactor.py", line 253, in run
    self._asyncioEventloop.run_forever()
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "C:\Users\dell\anaconda3\Lib\asyncio\base_events.py", line 680, in run_f

Scraping in progress...

Successfully captured 0 authors:
[]


C:\Users\dell\anaconda3\Lib\site-packages\scrapy\core\spidermw.py:433: ScrapyDeprecationWarning: __main__.SimpleSpider defines the deprecated start_requests() method. start_requests() has been deprecated in favor of a new method, start(), to support asynchronous code execution. start_requests() will stop being called in a future version of Scrapy. If you use Scrapy 2.13 or higher only, replace start_requests() with start(); note that start() is a coroutine (async def). If you need to maintain compatibility with lower Scrapy versions, when overriding start_requests() in a spider class, override start() as well; you can use super() to reuse the inherited start() implementation without copy-pasting. See the release notes of Scrapy 2.13 for details: https://docs.scrapy.org/en/2.13/news.html
  warn(
